# Stage 2: Demand Forecasting (from Stage 1 checkpoint)
**Input**: `demand.parquet` from Stage 1 (with `sale_amount_pred` column)  
**Method**: LightGBM, XGBoost, RF, Ridge, kNN, LSTM, N-HiTS + Stacking  
**Estimated runtime**: 4-5h on Kaggle T4 GPU


## 0. Setup & Dependencies


In [1]:

# Install NeuralForecast if missing
import subprocess, sys
try:
    import neuralforecast
except ImportError:
    import logging
    logging.info("Installing neuralforecast...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neuralforecast"])

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time
import xgboost as xgb
import lightgbm as lgb
import joblib

# Sklearn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# NeuralForecast
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM as NF_LSTM, NHITS, TimesNet, PatchTST
from neuralforecast.losses.pytorch import MAE


warnings.filterwarnings('ignore')

T0 = time.time()
def elapsed_h(): return (time.time() - T0) / 3600

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger(__name__)

# Install deps — pin numpy/pandas for neuralforecast compatibility
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "numpy==1.26.4", "pandas==2.2.2"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "lightgbm", "xgboost", "shap", "neuralforecast",
                       "datasets", "pyarrow", "joblib"])

import matplotlib
matplotlib.use("Agg")
OUT = Path("/kaggle/working")
CKPT = OUT / "checkpoints"
RESULTS = OUT / "results"
CKPT.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 96.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but yo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 72.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


## 1. Load Stage 1 Checkpoint
Load `demand.parquet` produced by Stage 1 (hourly imputation).  
This file contains `sale_amount_pred` = daily sum of imputed hourly sales.


In [2]:
# Load Stage 1 Checkpoint (demand.parquet)
import os
import glob
from pathlib import Path

# Search recursively for demand.parquet inside /kaggle/input/
log.info("Searching recursively for demand.parquet in /kaggle/input/...")
candidates = glob.glob("/kaggle/input/**/demand.parquet", recursive=True)
if candidates:
    DEMAND_PATH = Path(candidates[0])
    log.info(f"Loading demand from: {DEMAND_PATH}")
else:
    # Fallback to local
    DEMAND_PATH = Path("./demand.parquet")
    if not DEMAND_PATH.exists():
        log.error("No demand.parquet found! Please check if Stage 1 output is attached.")
        raise FileNotFoundError("demand.parquet not found")

df = pd.read_parquet(DEMAND_PATH)
df["dt"] = pd.to_datetime(df["dt"])
df["series_id"] = df["store_id"].astype(str) + "__" + df["product_id"].astype(str)

# Load eval data
eval_candidates = glob.glob("/kaggle/input/**/eval_data.parquet", recursive=True)
if eval_candidates:
    EVAL_PATH = Path(eval_candidates[0])
    log.info(f"Loading eval data from: {EVAL_PATH}")
    df_eval = pd.read_parquet(EVAL_PATH)
else:
    log.warning("eval_data.parquet not found, attempting to load from HuggingFace...")
    from datasets import load_dataset
    ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
    df_eval = (ds.get('eval') or ds.get('validation')).to_pandas()

df_eval["dt"] = pd.to_datetime(df_eval["dt"])
df_eval["series_id"] = df_eval["store_id"].astype(str) + "__" + df_eval["product_id"].astype(str)

# Verify sale_amount_pred exists
assert "sale_amount_pred" in df.columns, "sale_amount_pred not found! Are you sure this is Stage 1 output?"

# Stockout features
OPERATING_HOURS = 16
df["is_stockout"] = (df["stock_hour6_22_cnt"] > 0).astype(int)
df["stockout_ratio"] = df["stock_hour6_22_cnt"] / OPERATING_HOURS
df["stockout_hours"] = df["stock_hour6_22_cnt"]

log.info(f"Loaded {len(df):,} train rows, {len(df_eval):,} eval rows")
log.info(f"sale_amount_pred: mean={df['sale_amount_pred'].mean():.4f}, std={df['sale_amount_pred'].std():.4f}")
log.info(f"sale_amount:      mean={df['sale_amount'].mean():.4f}, std={df['sale_amount'].std():.4f}")

if "sale_amount" in df.columns and "sale_amount_pred" in df.columns:
    recovery_changed = int((np.abs(df['sale_amount_pred'] - df['sale_amount']) > 1e-6).sum())
    log.info(f"Recovery changed: {recovery_changed:,}/{len(df):,} ({recovery_changed/len(df)*100:.1f}%)")
log.info(f"Elapsed: {elapsed_h():.2f}h")

## 2. Feature Engineering


In [3]:
# ============================================================
log.info("=" * 60)
log.info("Feature Engineering")
log.info("=" * 60)

# Use sale_amount_pred (recovered) for lag/rolling features
# This debiases the features from censoring
df["y_eff"] = df["sale_amount_pred"]  # recovered demand, not censored

# Lag features (shift to prevent leakage — lag_k uses value from k days ago)
for lag in [1, 7, 14, 21, 28]:
    df[f"lag_{lag}"] = df.groupby("series_id")["y_eff"].shift(lag)

# Rolling features (shift(1) to prevent look-ahead)
for w in [7, 14, 28]:
    df[f"rolling_mean_{w}"] = df.groupby("series_id")["y_eff"].transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).mean())
    df[f"rolling_std_{w}"] = df.groupby("series_id")["y_eff"].transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).std())

# Stockout ratio rolling
for w in [7, 14, 28]:
    df[f"stockout_ratio_{w}d"] = df.groupby("series_id")["stockout_ratio"].transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).mean())

# Calendar features
df["day_of_week"] = df["dt"].dt.dayofweek
df["week_of_year"] = df["dt"].dt.isocalendar().week.astype(int)
df["month"] = df["dt"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["day_of_month"] = df["dt"].dt.day

# Fourier features for seasonality
for period in [7, 30]:
    t = (df["dt"] - df["dt"].min()).dt.days.values
    for k in [1, 2, 3]:
        df[f"sin_{period}d_h{k}"] = np.sin(2 * np.pi * k * t / period)
        df[f"cos_{period}d_h{k}"] = np.cos(2 * np.pi * k * t / period)

# Per-series scale (shift to prevent leakage)
df["series_scale"] = df.groupby("series_id")["y_eff"].transform(
    lambda x: x.shift(1).rolling(90, min_periods=7).mean())

# Recovery feature: ratio of recovered to observed demand
df["demand_ratio"] = df["sale_amount_pred"] / (df["sale_amount"] + 0.01)
df["demand_gap"] = df["sale_amount_pred"] - df["sale_amount"]

# Fill NaN numerics
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Clean up
df.drop(columns=["model_pred"], errors="ignore", inplace=True)
log.info(f"Features done: {len(df.columns)} columns. Elapsed: {elapsed_h():.2f}h")

## 3. Train / Validation / Test Split


In [4]:
# ============================================================
max_date = df["dt"].max()
# --- Add sample_weight and days_ago ---
days_ago = (max_date - df['dt']).dt.days
df['days_ago'] = days_ago
df['sample_weight'] = np.where(days_ago <= 30, 1.0, np.where(days_ago <= 60, 0.5, 0.25))

test_start = max_date - pd.Timedelta(days=6)
val_start = test_start - pd.Timedelta(days=7)
test = df[df["dt"] >= test_start].copy()
val = df[(df["dt"] >= val_start) & (df["dt"] < test_start)].copy()
train = df[df["dt"] < val_start].copy()
log.info(f"Split: train={len(train):,}, val={len(val):,}, test={len(test):,}")

# Feature columns — Step 4c: add stockout context directly [VN2 Winner]
# --- LEAKAGE FIX: Target Encoding (train-only) ---
log.info("Computing target encoding (train-only)...")
te_cols = ["first_category_id", "city_id"]
global_mean_train = train["y_eff"].mean()
smooth_factor = 100

for col in te_cols:
    te_map = train.groupby([col, "month"])["y_eff"].agg(["mean", "count"]).reset_index()
    te_map[f"{col}_te"] = (te_map["mean"] * te_map["count"] + global_mean_train * smooth_factor) / (te_map["count"] + smooth_factor)
    te_lookup = te_map[[col, "month", f"{col}_te"]]
    for split_df in [train, val, test]:
        split_df.drop(columns=[f"{col}_te"], errors="ignore", inplace=True)
        merged = split_df[[col, "month"]].merge(te_lookup, on=[col, "month"], how="left")
        split_df[f"{col}_te"] = merged[f"{col}_te"].fillna(global_mean_train).values

log.info(f"Target encoding (train-only): {[f'{c}_te' for c in te_cols]}")


FEATURE_COLS = [c for c in (
    [f"lag_{k}" for k in [1,7,14,21,28]] +
    [f"rolling_{s}_{w}" for s in ["mean","std"] for w in [7,14,28]] +
    [f"stockout_ratio_{w}d" for w in [7,14,28]] + ["stockout_ratio"] +
    ["is_stockout", "stockout_hours"] +  # Step 4c: direct stockout context
    ["series_scale"] +  # Step 4b: per-series scale factor as feature
    ["day_of_week","week_of_year","month","is_weekend","day_of_month"] +
    [f"{fn}_{p}d_h{k}" for p in [7,30] for k in [1,2,3] for fn in ["sin","cos"]] +
    ["discount","activity_flag","holiday_flag","precpt","avg_temperature","avg_humidity","avg_wind_level"] +
    ["first_category_id_te", "city_id_te"]  # target encoding
) if c in df.columns]

# Step 1: Use sale_amount_pred (recovered latent demand) as training target
# Paper methodology: train on recovered demand, evaluate on sale_amount (non-stockout only)
TARGET = "sale_amount_pred"

# Add city_id as integer feature for global models
if "city_id" not in FEATURE_COLS:
    FEATURE_COLS.append("city_id")
if "days_ago" not in FEATURE_COLS: FEATURE_COLS.append("days_ago")

log.info(f"Features: {len(FEATURE_COLS)}, Target: {TARGET}")
log.info(f"NOTE: Using global models (ablation proved global 28.24% < per-city 28.60%)")

## 6. XGBoost Global (Tuned)
Gradient boosted trees with early stopping. Produces SHAP feature importance analysis.
- 5000 estimators, max_depth=8, lr=0.02
- Early stopping on validation set (patience=100)


In [5]:
# ============================================================
log.info("=" * 60)
log.info("Training XGBoost GLOBAL (tuned)...")
log.info("=" * 60)

import xgboost as xgb

X_tr_xgb = train[FEATURE_COLS].values
y_tr_xgb = train[TARGET].values
w_tr_xgb = train["sample_weight"].values
X_val_xgb = val[FEATURE_COLS].values
y_val_xgb = val[TARGET].values
X_te_xgb = test[FEATURE_COLS].values
m_tr = ~np.isnan(y_tr_xgb)
m_v = ~np.isnan(y_val_xgb)

xgb_model = xgb.XGBRegressor(
    n_estimators=5000, max_depth=8, learning_rate=0.02,
    subsample=0.8, colsample_bytree=0.7, reg_alpha=0.05, reg_lambda=0.8,
    min_child_weight=10, tree_method="hist", random_state=42,
    early_stopping_rounds=100)
import glob
xgb_ckpt = []  # Disabled: retrain needed (target changed to sale_amount_pred)
if xgb_ckpt:
    log.info(f"Loading pre-trained XGBoost from {xgb_ckpt[0]}")
    xgb_model.load_model(xgb_ckpt[0])
else:
    xgb_model.fit(X_tr_xgb[m_tr], y_tr_xgb[m_tr], sample_weight=w_tr_xgb[m_tr],
              eval_set=[(X_val_xgb[m_v], y_val_xgb[m_v])], verbose=False)
    xgb_model.save_model(str(CKPT / "xgboost_global.json"))

xgb_val_preds = xgb_model.predict(X_val_xgb)
xgb_test_preds = xgb_model.predict(X_te_xgb)
log.info(f"XGBoost global: {getattr(xgb_model, 'best_iteration', 'loaded')} trees. Elapsed: {elapsed_h():.2f}h")

# SHAP analysis [Wu 2026]
log.info("Computing SHAP importance...")
import shap
X_shap = train[FEATURE_COLS].sample(min(5000, len(train)), random_state=42)
sv = shap.TreeExplainer(xgb_model).shap_values(X_shap)
importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "mean_abs_shap": np.abs(sv).mean(0)
}).sort_values("mean_abs_shap", ascending=False)
log.info(f"Top 10 SHAP:\n{importance.head(10).to_string()}")
importance.to_csv(RESULTS / "shap_importance.csv", index=False)

# SHAP bar chart
fig, ax = plt.subplots(figsize=(10, 8))
top20 = importance.head(20)
ax.barh(range(len(top20)), top20["mean_abs_shap"].values, color="steelblue")
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20["feature"].values)
ax.invert_yaxis()
ax.set_title("Top 20 Feature Importance (SHAP)", fontsize=14, fontweight="bold")
ax.set_xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.savefig(RESULTS / "shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()
log.info(f"SHAP done. Elapsed: {elapsed_h():.2f}h")

## 7. LightGBM Global (Tuned)
Two LightGBM models:
- **Model A (MSE)**: Trained on `sale_amount` → best WAPE
- **Model B (Recovered)**: Trained on `recovered_demand` → better WPE

Blend: 70% MSE + 30% Recovered.


In [6]:
# ============================================================
log.info("=" * 60)
log.info("Training LightGBM GLOBAL (tuned)...")
log.info("=" * 60)

import lightgbm as lgb

X_tr_lgb = train[FEATURE_COLS].values
y_tr_lgb = train[TARGET].values
w_tr_lgb = train["sample_weight"].values
X_val_lgb = val[FEATURE_COLS].values
y_val_lgb = val[TARGET].values
X_te_lgb = test[FEATURE_COLS].values
m_tr_l = ~np.isnan(y_tr_lgb)
m_v_l = ~np.isnan(y_val_lgb)

# Model A: MSE (best WAPE)
lgb_model = lgb.LGBMRegressor(
    n_estimators=8000, num_leaves=255, learning_rate=0.01,
    feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
    min_child_samples=50, reg_alpha=0.05, reg_lambda=1.0,
    random_state=42, verbose=-1)
import glob
lgb_ckpt = []  # Disabled: retrain needed
if lgb_ckpt:
    log.info(f"Loading pre-trained LightGBM (MSE) from {lgb_ckpt[0]}")
    lgb_model = lgb.Booster(model_file=lgb_ckpt[0])
    try: lgb_model.best_iteration_ = lgb_model.best_iteration
    except: lgb_model.best_iteration_ = "loaded"
else:
    lgb_model.fit(X_tr_lgb[m_tr_l], y_tr_lgb[m_tr_l], sample_weight=w_tr_lgb[m_tr_l],
                  eval_set=[(X_val_lgb[m_v_l], y_val_lgb[m_v_l])],
                  callbacks=[lgb.early_stopping(150, verbose=False)])
    lgb_model.booster_.save_model(str(CKPT / "lightgbm_global_mse.txt"))
log.info(f"LightGBM MSE: {lgb_model.best_iteration_} trees. Elapsed: {elapsed_h():.2f}h")

# Model B: LGB with explicit recovered target variable (same as TARGET now)
log.info("Training LightGBM on explicit sale_amount_pred target...")
y_tr_rec = train["sale_amount_pred"].values
y_val_rec = val["sale_amount_pred"].values
m_tr_rec = ~np.isnan(y_tr_rec)
m_v_rec = ~np.isnan(y_val_rec)

lgb_rec = lgb.LGBMRegressor(
    n_estimators=8000, num_leaves=255, learning_rate=0.01,
    feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
    min_child_samples=50, reg_alpha=0.05, reg_lambda=1.0,
    random_state=42, verbose=-1)
lgb_rec_ckpt = []  # Disabled: retrain needed
if lgb_rec_ckpt:
    log.info(f"Loading pre-trained LightGBM (Recovered) from {lgb_rec_ckpt[0]}")
    lgb_rec = lgb.Booster(model_file=lgb_rec_ckpt[0])
    try: lgb_rec.best_iteration_ = lgb_rec.best_iteration
    except: lgb_rec.best_iteration_ = "loaded"
else:
    lgb_rec.fit(X_tr_lgb[m_tr_rec], y_tr_rec[m_tr_rec], sample_weight=w_tr_lgb[m_tr_rec],
                eval_set=[(X_val_lgb[m_v_rec], y_val_rec[m_v_rec])],
                callbacks=[lgb.early_stopping(150, verbose=False)])
    lgb_rec.booster_.save_model(str(CKPT / "lightgbm_global_recovered.txt"))
log.info(f"LightGBM Recovered: {lgb_rec.best_iteration_} trees. Elapsed: {elapsed_h():.2f}h")

# Predictions from both (evaluated against sale_amount)
lgb_mse_val = lgb_model.predict(X_val_lgb)
lgb_mse_test = lgb_model.predict(X_te_lgb)
lgb_rec_val = lgb_rec.predict(X_val_lgb)
lgb_rec_test = lgb_rec.predict(X_te_lgb)

# Blend 70% MSE (best WAPE) + 30% Recovered (best WPE)
BLEND_MSE, BLEND_REC = 0.7, 0.3
lgb_val_preds = BLEND_MSE * lgb_mse_val + BLEND_REC * lgb_rec_val
lgb_test_preds = BLEND_MSE * lgb_mse_test + BLEND_REC * lgb_rec_test

# Report individual + blend metrics on val (paper: vs sale_amount, non-stockout only)
val_non_oos = (val["is_stockout"].values == 0)
m_eval_lgb = m_v_l & val_non_oos
y_v = val["sale_amount"].values[m_eval_lgb]
for name, p in [("MSE-target", lgb_mse_val[m_eval_lgb]), ("Recovered-target", lgb_rec_val[m_eval_lgb]),
                (f"Blend {BLEND_MSE:.0%}+{BLEND_REC:.0%}", lgb_val_preds[m_eval_lgb])]:
    w = np.sum(np.abs(p - y_v)) / np.sum(np.abs(y_v)) * 100
    wp = (np.sum(p) - np.sum(y_v)) / np.sum(np.abs(y_v)) * 100
    log.info(f"  LGB {name}: WAPE={w:.2f}%, WPE={wp:.2f}%")

## 7b. Random Forest (sklearn)
**Algorithm family: Random Forest**

500 trees, max_depth=12, min_samples_leaf=20. Verbose output to track progress on large dataset (~4.5M rows).


In [7]:
# ============================================================
log.info("=" * 60)
log.info("Training Random Forest...")
log.info("=" * 60)

from sklearn.ensemble import RandomForestRegressor

import glob, joblib as _jl
rf_ckpt = []  # Disabled: retrain needed

m_tr_rf = ~np.isnan(train[TARGET].values)

if rf_ckpt:
    log.info(f"Loading pre-trained Random Forest from {rf_ckpt[0]}")
    rf_model = _jl.load(rf_ckpt[0])
else:
    rf_model = RandomForestRegressor(
        n_estimators=500, max_depth=12, min_samples_leaf=20,
        max_features=0.7, n_jobs=-1, random_state=42, verbose=2)
    rf_model.fit(train[FEATURE_COLS].values[m_tr_rf], train[TARGET].values[m_tr_rf],
                 sample_weight=train["sample_weight"].values[m_tr_rf])
    _jl.dump(rf_model, str(CKPT / "random_forest_global.joblib"))

rf_val_preds = rf_model.predict(val[FEATURE_COLS].values)
rf_test_preds = rf_model.predict(test[FEATURE_COLS].values)
log.info(f"Random Forest: {rf_model.n_estimators} trees. Elapsed: {elapsed_h():.2f}h")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


building tree 1 of 500
building tree 2 of 500
building tree 3 of 500
building tree 4 of 500
building tree 5 of 500
building tree 6 of 500
building tree 7 of 500
building tree 8 of 500
building tree 9 of 500
building tree 10 of 500
building tree 11 of 500
building tree 12 of 500
building tree 13 of 500
building tree 14 of 500
building tree 15 of 500
building tree 16 of 500
building tree 17 of 500
building tree 18 of 500
building tree 19 of 500
building tree 20 of 500
building tree 21 of 500
building tree 22 of 500
building tree 23 of 500
building tree 24 of 500
building tree 25 of 500
building tree 26 of 500
building tree 27 of 500
building tree 28 of 500
building tree 29 of 500
building tree 30 of 500
building tree 31 of 500
building tree 32 of 500
building tree 33 of 500
building tree 34 of 500
building tree 35 of 500
building tree 36 of 500
building tree 37 of 500


[Parallel(n_jobs=-1)]: Done  33 tasks      | elapsed: 14.3min


building tree 38 of 500
building tree 39 of 500
building tree 40 of 500
building tree 41 of 500
building tree 42 of 500
building tree 43 of 500
building tree 44 of 500
building tree 45 of 500
building tree 46 of 500
building tree 47 of 500
building tree 48 of 500
building tree 49 of 500
building tree 50 of 500
building tree 51 of 500
building tree 52 of 500
building tree 53 of 500
building tree 54 of 500
building tree 55 of 500
building tree 56 of 500
building tree 57 of 500
building tree 58 of 500
building tree 59 of 500
building tree 60 of 500
building tree 61 of 500
building tree 62 of 500
building tree 63 of 500
building tree 64 of 500
building tree 65 of 500
building tree 66 of 500
building tree 67 of 500
building tree 68 of 500
building tree 69 of 500
building tree 70 of 500
building tree 71 of 500
building tree 72 of 500
building tree 73 of 500
building tree 74 of 500
building tree 75 of 500
building tree 76 of 500
building tree 77 of 500
building tree 78 of 500
building tree 79

[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed: 62.5min


building tree 159 of 500
building tree 160 of 500
building tree 161 of 500
building tree 162 of 500
building tree 163 of 500
building tree 164 of 500
building tree 165 of 500
building tree 166 of 500
building tree 167 of 500
building tree 168 of 500
building tree 169 of 500
building tree 170 of 500
building tree 171 of 500
building tree 172 of 500
building tree 173 of 500
building tree 174 of 500
building tree 175 of 500
building tree 176 of 500
building tree 177 of 500
building tree 178 of 500
building tree 179 of 500
building tree 180 of 500
building tree 181 of 500
building tree 182 of 500
building tree 183 of 500
building tree 184 of 500
building tree 185 of 500
building tree 186 of 500
building tree 187 of 500
building tree 188 of 500
building tree 189 of 500
building tree 190 of 500
building tree 191 of 500
building tree 192 of 500
building tree 193 of 500
building tree 194 of 500
building tree 195 of 500
building tree 196 of 500
building tree 197 of 500
building tree 198 of 500


[Parallel(n_jobs=-1)]: Done 357 tasks      | elapsed: 143.3min


building tree 362 of 500
building tree 363 of 500
building tree 364 of 500
building tree 365 of 500
building tree 366 of 500
building tree 367 of 500
building tree 368 of 500
building tree 369 of 500
building tree 370 of 500
building tree 371 of 500
building tree 372 of 500
building tree 373 of 500
building tree 374 of 500
building tree 375 of 500
building tree 376 of 500
building tree 377 of 500
building tree 378 of 500
building tree 379 of 500
building tree 380 of 500
building tree 381 of 500
building tree 382 of 500
building tree 383 of 500
building tree 384 of 500
building tree 385 of 500
building tree 386 of 500
building tree 387 of 500
building tree 388 of 500
building tree 389 of 500
building tree 390 of 500
building tree 391 of 500
building tree 392 of 500
building tree 393 of 500
building tree 394 of 500
building tree 395 of 500
building tree 396 of 500
building tree 397 of 500
building tree 398 of 500
building tree 399 of 500
building tree 400 of 500
building tree 401 of 500


[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed: 200.1min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:    1.9s
[Parallel(n_jobs=4)]: Done 357 tasks      | elapsed:    4.4s
[Parallel(n_jobs=4)]: Done 500 out of 500 | elapsed:    6.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:    2.0s
[Parallel(n_jobs=4)]: Done 357 tasks      | elapsed:    4.5s
[Parallel(n_jobs=4)]: Done 500 out of 500 | elapsed:    6.2s finished


## 7c. Ridge Regression (sklearn)
**Algorithm family: Linear Regression / Ridge**

Standalone Ridge (α=10) with StandardScaler + SimpleImputer (median) preprocessing. Serves as linear baseline.


In [8]:
# ============================================================
log.info("=" * 60)
log.info("Training Ridge Regression (standalone)...")
log.info("=" * 60)

from sklearn.linear_model import Ridge as RidgeBase
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Ridge/kNN need clean features — impute NaN then scale
ridge_imputer = SimpleImputer(strategy="median")
ridge_scaler = StandardScaler()
X_tr_ridge = ridge_scaler.fit_transform(ridge_imputer.fit_transform(train[FEATURE_COLS].values[m_tr_rf]))
X_val_ridge = ridge_scaler.transform(ridge_imputer.transform(val[FEATURE_COLS].values))
X_te_ridge = ridge_scaler.transform(ridge_imputer.transform(test[FEATURE_COLS].values))

import glob, joblib as _jl
ridge_ckpt = []  # Disabled: retrain needed
if ridge_ckpt:
    log.info(f"Loading pre-trained Ridge from {ridge_ckpt[0]}")
    ckpt_data = _jl.load(ridge_ckpt[0])
    ridge_standalone = ckpt_data["model"]
    ridge_scaler = ckpt_data["scaler"]
else:
    ridge_standalone = RidgeBase(alpha=10.0)
    ridge_standalone.fit(X_tr_ridge, train[TARGET].values[m_tr_rf],
                     sample_weight=train["sample_weight"].values[m_tr_rf])
ridge_val_preds = ridge_standalone.predict(X_val_ridge)
ridge_test_preds = ridge_standalone.predict(X_te_ridge)
if not ridge_ckpt:
    _jl.dump({"model": ridge_standalone, "scaler": ridge_scaler}, str(CKPT / "ridge_standalone.joblib"))
log.info(f"Ridge standalone done. Coefficients: {np.count_nonzero(ridge_standalone.coef_)}/{len(ridge_standalone.coef_)} non-zero. Elapsed: {elapsed_h():.2f}h")

## 7d. kNN Regressor (sklearn)
**Algorithm family: k-Nearest Neighbors**

k=15, distance-weighted, reuses the imputed+scaled features from Ridge. Instance-based baseline.


In [9]:
# ============================================================
log.info("=" * 60)
log.info("Training kNN Regressor...")
log.info("=" * 60)

from sklearn.neighbors import KNeighborsRegressor

# kNN needs scaled features; reuse ridge_scaler
import glob, joblib as _jl
knn_ckpt = []  # Disabled: retrain needed

if knn_ckpt:
    log.info(f"Loading pre-trained kNN from {knn_ckpt[0]}")
    knn_model = _jl.load(knn_ckpt[0])
else:
    knn_model = KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=-1, algorithm="auto")
    log.info("kNN fitting...")
    knn_model.fit(X_tr_ridge, train[TARGET].values[m_tr_rf])
    _jl.dump(knn_model, str(CKPT / "knn_global.joblib"))

log.info("kNN predicting val...")
knn_val_preds = knn_model.predict(X_val_ridge)
log.info("kNN predicting test...")
knn_test_preds = knn_model.predict(X_te_ridge)
log.info(f"kNN (k={knn_model.n_neighbors}) done. Elapsed: {elapsed_h():.2f}h")

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7afef33107c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error


## 7e. LSTM (NeuralForecast)
**Algorithm family: LSTM / GRU**

2-layer LSTM encoder (hidden=128), cross-validation aligned to val/test dates. 1000 training steps on GPU.


In [10]:
# ============================================================
log.info("=" * 60)
log.info("Training LSTM...")
log.info("=" * 60)

from neuralforecast.models import LSTM as NF_LSTM

lstm_val_preds = np.full(len(val), np.nan)
lstm_test_preds = np.full(len(test), np.nan)

nf_lstm_target = df[["series_id", "dt", TARGET]].copy()
nf_lstm_target.columns = ["unique_id", "ds", "y"]
nf_lstm_target["y"] = nf_lstm_target["y"].fillna(0)

lstm_model = NF_LSTM(h=7, input_size=30, encoder_hidden_size=128, encoder_n_layers=2,
                     decoder_hidden_size=128, loss=MAE(),
                     max_steps=1000, batch_size=32, learning_rate=1e-3,
                     scaler_type="robust", random_seed=42, accelerator="gpu", devices=1)

nf_lstm = NeuralForecast(models=[lstm_model], freq="D")

try:
    cv_lstm = nf_lstm.cross_validation(df=nf_lstm_target, n_windows=2, step_size=7).reset_index()
    cv_lstm["ds"] = pd.to_datetime(cv_lstm["ds"])
    lstm_col = [c for c in cv_lstm.columns if "LSTM" in c][0]

    for split_name, split_df, pred_arr in [("val", val, lstm_val_preds), ("test", test, lstm_test_preds)]:
        m = split_df[["series_id","dt"]].merge(
            cv_lstm.rename(columns={"unique_id":"series_id","ds":"dt"}),
            on=["series_id","dt"], how="left")
        if lstm_col in m.columns:
            mask = ~m[lstm_col].isna()
            pred_arr[mask.values] = m.loc[mask, lstm_col].values
            log.info(f"LSTM aligned {split_name}: {mask.sum():,}/{len(split_df):,}")
except Exception as e:
    log.error(f"LSTM CV failed: {e}")
    nf_lstm.fit(df=nf_lstm_target)

log.info(f"LSTM done. Elapsed: {elapsed_h():.2f}h")
del nf_lstm, lstm_model; torch.cuda.empty_cache()

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
2026-05-24 17:33:41.817402: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779644022.218555      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779644022.346970      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779644023.344432      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779644023.344459      23 comput

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1000` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

## 8. N-HiTS (NeuralForecast)
**Algorithm family: Neural (hierarchical interpolation)**

Modern neural forecasting model with multi-rate signal decomposition. 2.4M parameters, 1000 training steps.


In [11]:
# ============================================================
log.info("=" * 60)
log.info("Training N-HiTS...")
log.info("=" * 60)

from neuralforecast.models import NHITS

nhits_val_preds = np.full(len(val), np.nan)
nhits_test_preds = np.full(len(test), np.nan)

nf_target = df[["series_id", "dt", TARGET]].copy()
nf_target.columns = ["unique_id", "ds", "y"]
nf_target["y"] = nf_target["y"].fillna(0)

nhits = NHITS(h=7, input_size=30, n_blocks=[1,1,1],
    mlp_units=[[512,512],[512,512],[512,512]], n_pool_kernel_size=[4,2,1],
    loss=MAE(), max_steps=1000, batch_size=32, learning_rate=1e-3, scaler_type="robust", random_seed=42, accelerator="gpu", devices=1)

nf_nh = NeuralForecast(models=[nhits], freq="D")

try:
    cv_nh = nf_nh.cross_validation(df=nf_target, n_windows=2, step_size=7).reset_index()
    cv_nh["ds"] = pd.to_datetime(cv_nh["ds"])
    nh_col = [c for c in cv_nh.columns if "NHITS" in c][0]

    for split_name, split_df, pred_arr in [("val", val, nhits_val_preds), ("test", test, nhits_test_preds)]:
        m = split_df[["series_id","dt"]].merge(
            cv_nh.rename(columns={"unique_id":"series_id","ds":"dt"}),
            on=["series_id","dt"], how="left")
        if nh_col in m.columns:
            mask = ~m[nh_col].isna()
            pred_arr[mask.values] = m.loc[mask, nh_col].values
            log.info(f"N-HiTS aligned {split_name}: {mask.sum():,}/{len(split_df):,}")
except Exception as e:
    log.error(f"N-HiTS CV failed: {e}")
    nf_nh.fit(df=nf_target)
    nf_nh.predict().to_parquet(OUT / "preds_nhits.parquet")

log.info(f"N-HiTS done. Elapsed: {elapsed_h():.2f}h")
del nf_nh, nhits; torch.cuda.empty_cache()

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.4 M  | train
-------------------------------------------------------
2.4 M     Trainable params
0         Non-trainable params
2.4 M     Total params
9.778     Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1000` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

## 9. Stacking Ensemble (Ridge Meta-Learner)
Stack XGBoost + LightGBM + Random Forest predictions via Ridge regression meta-learner.
- TimeSeriesSplit cross-validation (3 folds)
- Reports meta-weights showing each base model's contribution


In [12]:
# ============================================================
log.info("=" * 60)
log.info("Stacking ensemble (Ridge)")
log.info("=" * 60)

from sklearn.linear_model import Ridge

# Stack XGB + LGB + Random Forest — diverse model families for better ensemble
base_val = {"xgboost": xgb_val_preds, "lightgbm": lgb_val_preds, "random_forest": rf_val_preds}
base_test = {"xgboost": xgb_test_preds, "lightgbm": lgb_test_preds, "random_forest": rf_test_preds}
log.info(f"Stacking: XGB + LGB + RF (3 diverse tree-based models)")

log.info(f"Base models: {list(base_val.keys())}")

y_val = val[TARGET].values
valid = ~np.isnan(y_val)
for k in base_val: valid &= ~np.isnan(base_val[k])

if valid.sum() > 100:
    sorted_keys = sorted(base_val.keys())
    X_meta = np.column_stack([base_val[k][valid] for k in sorted_keys])
    y_meta = y_val[valid]

    # Cross-validated stacking performance
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    for tr_idx, vl_idx in tscv.split(X_meta):
        ridge_cv = Ridge(alpha=1.0).fit(X_meta[tr_idx], y_meta[tr_idx])
        fold_pred = ridge_cv.predict(X_meta[vl_idx])
        fold_rmse = np.sqrt(np.mean((y_meta[vl_idx] - fold_pred) ** 2))
        cv_scores.append(fold_rmse)
    log.info(f"Stacking CV RMSE: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

    # Final meta-learner
    meta = Ridge(alpha=1.0).fit(X_meta, y_meta)
    weights = dict(zip(sorted_keys, meta.coef_))
    log.info(f"Meta weights: {weights}")
    log.info(f"Meta intercept: {meta.intercept_:.4f}")

    import joblib; joblib.dump(meta, CKPT / "stacking_ridge.joblib")

    X_test_meta = np.column_stack([base_test[k] for k in sorted_keys])
    tv = ~np.isnan(X_test_meta).any(axis=1)
    stacking_test = np.full(len(test), np.nan)
    stacking_test[tv] = meta.predict(X_test_meta[tv])
else:
    stacking_test = lgb_test_preds

## 10. Ablation Study Models
Three controlled experiments:
1. **Recovery impact**: XGBoost on `recovered_demand` vs `sale_amount`
2. **Global vs per-city**: Single global model vs 18 city-specific models
3. **Fourier features**: With vs without Fourier harmonics


In [13]:
# ============================================================
log.info("=" * 60)
log.info("Training ablation models...")
log.info("=" * 60)

# Ablation 2: Global XGB on sale_amount_pred — proves recovery impact
log.info("Ablation: Global XGB on sale_amount_pred (with recovery)...")
y_rec = train["sale_amount_pred"].values
m_rec = ~np.isnan(y_rec)
xgb_rec_model = xgb.XGBRegressor(n_estimators=2000, max_depth=8, learning_rate=0.03,
                                  tree_method="hist", random_state=42)
xgb_rec_model.fit(train[FEATURE_COLS].values[m_rec], y_rec[m_rec], verbose=False)
xgb_recovered_test = xgb_rec_model.predict(test[FEATURE_COLS].values)
log.info(f"Ablation XGB-recovered done. Elapsed: {elapsed_h():.2f}h")

# Ablation 4: Per-city XGBoost — validates global > per-city
log.info("Ablation: Per-city XGBoost...")
xgb_percity_test = np.full(len(test), np.nan)
for city in sorted(train["city_id"].unique()):
    ct, cte = train[train["city_id"]==city], test[test["city_id"]==city]
    if len(ct) < 100: continue
    X_tr, y_tr = ct[FEATURE_COLS].values, ct[TARGET].values
    m_tr = ~np.isnan(y_tr)
    if m_tr.sum() < 50: continue
    m = xgb.XGBRegressor(n_estimators=2000, max_depth=7, learning_rate=0.03,
                          tree_method="hist", random_state=42)
    m.fit(X_tr[m_tr], y_tr[m_tr], verbose=False)
    tm = test["city_id"]==city
    if tm.sum() > 0: xgb_percity_test[tm.values] = m.predict(cte[FEATURE_COLS].values)
log.info(f"Ablation XGB-perCity done. Elapsed: {elapsed_h():.2f}h")

# Ablation 5: Global XGB without Fourier features
log.info("Ablation: Global XGB without Fourier features...")
NO_FOURIER = [c for c in FEATURE_COLS if not c.startswith("sin_") and not c.startswith("cos_")]
m_nof = ~np.isnan(train[TARGET].values)
xgb_nof_model = xgb.XGBRegressor(n_estimators=2000, max_depth=8, learning_rate=0.03,
                                   tree_method="hist", random_state=42)
xgb_nof_model.fit(train[NO_FOURIER].values[m_nof], train[TARGET].values[m_nof], verbose=False)
xgb_nofourier_test = xgb_nof_model.predict(test[NO_FOURIER].values)
log.info(f"Ablation XGB-noFourier done. Elapsed: {elapsed_h():.2f}h")

## 11. Benchmark Evaluation
Evaluate all 10 models on the held-out test set:
- Metrics: RMSE, MAE, R², WAPE, WPE, ρ_DS (stockout decoupling score)
- Compare against FreshRetailNet-50K paper baseline (WAPE=27.62%, R²=0.816)


In [14]:
# ============================================================
log.info("=" * 60)
log.info("BENCHMARK EVALUATION")
log.info("=" * 60)

def eval_metrics(y_true, y_pred, name, stockout_mask=None):
    """Full evaluation: RMSE, MAE, R², WAPE, WPE, ρ_DS.
    Paper methodology: evaluate against sale_amount on NON-STOCKOUT days only."""
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    yt, yp = y_true[mask], y_pred[mask]
    if len(yt) == 0:
        return {"model": name, "n": 0}
    total = np.sum(np.abs(yt))
    ss_res = np.sum((yt - yp)**2)
    ss_tot = np.sum((yt - yt.mean())**2)
    result = {
        "model": name, "n": int(len(yt)),
        "RMSE": float(np.sqrt(np.mean((yt - yp)**2))),
        "MAE": float(np.mean(np.abs(yt - yp))),
        "R2": float(1 - ss_res / max(ss_tot, 1e-8)),
        "WAPE": float(np.sum(np.abs(yt - yp)) / max(total, 1e-8) * 100),
        "WPE": float((np.sum(yp) - np.sum(yt)) / max(total, 1e-8) * 100),
    }
    # Decoupling score ρ_DS [FRN-50K paper]
    if stockout_mask is not None:
        sm = stockout_mask[mask]
        errors = np.abs(yt - yp)
        if sm.std() > 0 and errors.std() > 0:
            result["rho_DS"] = float(np.corrcoef(errors, sm)[0, 1])
        else:
            result["rho_DS"] = 0.0
    return result

# Paper methodology: evaluate against sale_amount on NON-STOCKOUT days only
# On stockout days, sale_amount is censored and not true demand
non_stockout_mask = (test["is_stockout"].values == 0)
y_test = test["sale_amount"].values.copy()
y_test[~non_stockout_mask] = np.nan  # mask out stockout days from evaluation
stockout_test = test["is_stockout"].values
log.info(f"Evaluation: {non_stockout_mask.sum():,} non-stockout / {len(test):,} total test rows")

# Main models
results = [
    eval_metrics(y_test, xgb_test_preds, "XGBoost (global)", stockout_test),
    eval_metrics(y_test, lgb_mse_test, "LGB-MSE (global)", stockout_test),
    eval_metrics(y_test, lgb_rec_test, "LGB-Recovered (global)", stockout_test),
    eval_metrics(y_test, lgb_test_preds, "LGB Blend 70/30", stockout_test),
    eval_metrics(y_test, rf_test_preds, "Random Forest (global)", stockout_test),
    eval_metrics(y_test, ridge_test_preds, "Ridge Regression", stockout_test),
    eval_metrics(y_test, knn_test_preds, "kNN (k=15)", stockout_test),
]
if not np.all(np.isnan(lstm_test_preds)):
    results.append(eval_metrics(y_test, lstm_test_preds, "LSTM (global)", stockout_test))
if not np.all(np.isnan(nhits_test_preds)):
    results.append(eval_metrics(y_test, nhits_test_preds, "N-HiTS (global)", stockout_test))
results.append(eval_metrics(y_test, stacking_test, "Stacking Ensemble", stockout_test))

# Ablation models — all evaluated against sale_amount
results_ablation = [
    eval_metrics(y_test, xgb_recovered_test, "XGB recovered target", stockout_test),
    eval_metrics(y_test, xgb_percity_test, "XGB per-city", stockout_test),
    eval_metrics(y_test, xgb_nofourier_test, "XGB no Fourier", stockout_test),
]

# Print main results
print("\n" + "=" * 95)
print("MAIN BENCHMARK RESULTS (evaluated against sale_amount, non-stockout days only)")
print("=" * 95)
print(f"{'Model':<30} {'N':>8} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'WAPE%':>8} {'WPE%':>8} {'ρ_DS':>6}")
print("-" * 95)
for r in results:
    if r.get("n", 0) > 0:
        rho = r.get("rho_DS", 0)
        print(f"{r['model']:<30} {r['n']:>8,} {r['RMSE']:>8.4f} {r['MAE']:>8.4f} "
              f"{r['R2']:>8.4f} {r['WAPE']:>8.2f} {r['WPE']:>8.2f} {rho:>6.3f}")
print("=" * 95)

# Print ablation results
print("\nABLATION RESULTS")
print("-" * 95)
for r in results_ablation:
    if r.get("n", 0) > 0:
        rho = r.get("rho_DS", 0)
        print(f"{r['model']:<30} {r['n']:>8,} {r['RMSE']:>8.4f} {r['MAE']:>8.4f} "
              f"{r['R2']:>8.4f} {r['WAPE']:>8.2f} {r['WPE']:>8.2f} {rho:>6.3f}")
print("=" * 95)

# Ablation analysis
best = min([r for r in results if r.get("n", 0) > 0], key=lambda x: x["WAPE"])
xgb_main = [r for r in results if "XGBoost" in r["model"]][0]
xgb_rec = results_ablation[0]   # recovered target
xgb_pc = results_ablation[1]    # per-city
xgb_nof = results_ablation[2]   # no fourier

print("\n📊 ABLATION ANALYSIS")
print(f"  Ablation 2 (recovery impact):    raw {xgb_main['WAPE']:.2f}% vs recovered {xgb_rec['WAPE']:.2f}% "
      f"(Δ = {xgb_rec['WAPE'] - xgb_main['WAPE']:+.2f}%)")
print(f"  Ablation 3 (stacking vs single): WAPE {xgb_main['WAPE']:.2f}% → {best['WAPE']:.2f}% "
      f"(Δ = {xgb_main['WAPE'] - best['WAPE']:+.2f}%)")
print(f"  Ablation 4 (global vs per-city): WAPE {xgb_main['WAPE']:.2f}% vs {xgb_pc['WAPE']:.2f}% "
      f"(Δ = {xgb_pc['WAPE'] - xgb_main['WAPE']:+.2f}%)")
print(f"  Ablation 5 (Fourier features):   WAPE {xgb_nof['WAPE']:.2f}% → {xgb_main['WAPE']:.2f}% "
      f"(Δ = {xgb_nof['WAPE'] - xgb_main['WAPE']:+.2f}%)")

print(f"\nPaper baseline: WAPE=27.62%, R²=0.816, ρ_DS=0.07")
print(f"Our target:     WAPE<25%,    R²>0.95,  ρ_DS<0.05")
print(f"\n🏆 Best model: {best['model']} (WAPE={best['WAPE']:.2f}%, R²={best['R2']:.4f}, "
      f"ρ_DS={best.get('rho_DS', 0):.3f})")
if best["WAPE"] < 27.62:
    print("✅ BEAT paper baseline!")
elif best["WAPE"] < 30:
    print("⚠️ Close to baseline, tuning may help")
else:
    print("❌ Below baseline, further optimization needed")


MAIN BENCHMARK RESULTS (evaluated against sale_amount, non-stockout days only)
Model                                 N     RMSE      MAE       R²    WAPE%     WPE%   ρ_DS
-----------------------------------------------------------------------------------------------
XGBoost (global)                198,987   0.6395   0.3410   0.8785    28.89    -3.91  0.000
LGB-MSE (global)                198,987   0.6128   0.3390   0.8885    28.73    -5.90  0.000
LGB-Recovered (global)          198,987   0.6128   0.3390   0.8885    28.73    -5.90  0.000
LGB Blend 70/30                 198,987   0.6128   0.3390   0.8885    28.73    -5.90  0.000
Random Forest (global)          198,987   0.5184   0.3330   0.9202    28.21    -3.50  0.000
Ridge Regression                198,987   0.5255   0.3500   0.9179    29.66    -7.89  0.000
kNN (k=15)                      198,987   0.6474   0.4036   0.8755    34.20   -14.47  0.000
LSTM (global)                   198,987   0.6036   0.3766   0.8918    31.91    -1.46  0.

## 12. Per-City Breakdown
Break down best model's performance across all 18 cities to identify geographic patterns.


In [15]:
# ============================================================
log.info("=" * 60)
log.info("PER-CITY BREAKDOWN (best model)")
log.info("=" * 60)

# Use best individual model's predictions for per-city
preds_map = {
    "XGBoost (global)": xgb_test_preds,
    "LGB-MSE (global)": lgb_mse_test,
    "LGB-Recovered (global)": lgb_rec_test,
    "LGB Blend 70/30": lgb_test_preds,
    "Random Forest (global)": rf_test_preds,
    "Ridge Regression": ridge_test_preds,
    "kNN (k=15)": knn_test_preds,
    "LSTM (global)": lstm_test_preds,
    "N-HiTS (global)": nhits_test_preds,
    "Stacking Ensemble": stacking_test,
}
best_preds = preds_map.get(best["model"], lgb_mse_test)

city_results = []
for city in sorted(test["city_id"].unique()):
    cm = test["city_id"] == city
    if cm.sum() == 0: continue
    # Use same non-stockout filtered y_test
    cr = eval_metrics(y_test[cm.values], best_preds[cm.values],
                      f"City {city}", stockout_test[cm.values])
    if cr.get("n", 0) > 0:
        city_results.append(cr)

print("\n" + "=" * 95)
print(f"PER-CITY BREAKDOWN ({best['model']})")
print("=" * 95)
print(f"{'City':<30} {'N':>8} {'RMSE':>8} {'WAPE%':>8} {'WPE%':>8} {'ρ_DS':>6}")
print("-" * 95)
for cr in sorted(city_results, key=lambda x: x["WAPE"]):
    print(f"{cr['model']:<30} {cr['n']:>8,} {cr['RMSE']:>8.4f} "
          f"{cr['WAPE']:>8.2f} {cr['WPE']:>8.2f} {cr.get('rho_DS', 0):>6.3f}")
print("=" * 95)


PER-CITY BREAKDOWN (Stacking Ensemble)
City                                  N     RMSE    WAPE%     WPE%   ρ_DS
-----------------------------------------------------------------------------------------------
City 14                           2,086   0.4680    22.45    -0.82  0.000
City 4                            4,550   0.6169    23.23    -4.40  0.000
City 11                           7,014   0.6581    25.15    -3.64  0.000
City 16                          17,516   0.5609    26.48    -5.86  0.000
City 6                           10,000   0.5601    27.24    -9.51  0.000
City 15                           2,573   0.4968    27.38    -6.78  0.000
City 17                           1,098   0.6167    27.77     1.23  0.000
City 12                          17,883   0.4920    28.05    -8.78  0.000
City 0                          105,710   0.5143    28.08    -5.25  0.000
City 5                            2,447   0.5302    28.47    -7.38  0.000
City 13                           9,546   0.4371  

## 13. Save All Results
Export benchmark results to JSON and generate markdown report table.


In [16]:
import json
# ============================================================

all_results = {
    "main": results,
    "ablations": results_ablation,
    "per_city": city_results,
    "best_model": best["model"],
    "pipeline_hours": elapsed_h(),
    "recovery_changed_pct": recovery_changed / len(df) * 100,
}
with open(RESULTS / "benchmark_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

# Generate comprehensive markdown report
md = ["# FreshRetailNet-50K Benchmark Results\n",
      f"**Pipeline completed in {elapsed_h():.2f}h**\n",
      "## Task 2: Demand Forecasting\n",
      "| Model | N | RMSE | MAE | R² | WAPE (%) | WPE (%) | ρ_DS |",
      "|-------|---|------|-----|----|----------|---------|------|"]
for r in results:
    if r.get("n", 0) > 0:
        md.append(f"| {r['model']} | {r['n']:,} | {r['RMSE']:.4f} | {r['MAE']:.4f} | "
                  f"{r['R2']:.4f} | {r['WAPE']:.2f} | {r['WPE']:.2f} | {r.get('rho_DS',0):.3f} |")

md.append(f"\n**Paper baseline:** WAPE=27.62%, R²=0.816, ρ_DS=0.07")
md.append(f"**Best model:** {best['model']} (WAPE={best['WAPE']:.2f}%)\n")

md.append("## Ablation Study\n")
md.append("| Ablation | Condition | WAPE (%) | Δ WAPE |")
md.append("|----------|-----------|----------|--------|")
md.append(f"| Recovery impact | raw vs recovered | {xgb_main['WAPE']:.2f} vs {xgb_rec['WAPE']:.2f} | {xgb_rec['WAPE']-xgb_main['WAPE']:+.2f} |")
md.append(f"| Stacking value | single → ensemble | {xgb_main['WAPE']:.2f} → {best['WAPE']:.2f} | {xgb_main['WAPE']-best['WAPE']:+.2f} |")
md.append(f"| Global vs per-city | global vs per-city | {xgb_main['WAPE']:.2f} vs {xgb_pc['WAPE']:.2f} | {xgb_pc['WAPE']-xgb_main['WAPE']:+.2f} |")
md.append(f"| Fourier features | without → with | {xgb_nof['WAPE']:.2f} → {xgb_main['WAPE']:.2f} | {xgb_nof['WAPE']-xgb_main['WAPE']:+.2f} |")

if (RESULTS / "task1_metrics.json").exists():
    t1 = json.load(open(RESULTS / "task1_metrics.json"))
    md.append(f"\n## Task 1: Latent Demand Recovery\n")
    md.append(f"| Metric | Value |")
    md.append(f"|--------|-------|")
    md.append(f"| Ensemble WAPE | {t1.get('WAPE', 'N/A'):.2f}% |")
    md.append(f"| Ensemble RMSE | {t1.get('RMSE', 'N/A'):.4f} |")
    md.append(f"| TimesNet weight | {t1.get('w_timesnet', 'N/A'):.3f} |")
    md.append(f"| PatchTST weight | {t1.get('w_patchtst', 'N/A'):.3f} |")
    md.append(f"| Records recovered | {recovery_changed:,} ({recovery_changed/len(df)*100:.1f}%) |")

with open(RESULTS / "benchmark_table.md", "w") as f:
    f.write("\n".join(md))

## 14. Session Summary
Print final pipeline statistics: output files, recovery rate, best model performance.


In [17]:
# ============================================================
log.info("=" * 60)
log.info(f"PIPELINE COMPLETE — Total time: {elapsed_h():.2f}h")
log.info("=" * 60)
log.info(f"Output files in {OUT}:")
for f in sorted(OUT.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        log.info(f"  {f.relative_to(OUT)}: {size_mb:.1f} MB")

log.info(f"\nRecovery: {recovery_changed:,}/{len(df):,} rows ({recovery_changed/len(df)*100:.1f}%)")
log.info(f"Best: {best['model']} → WAPE={best['WAPE']:.2f}%, R²={best['R2']:.4f}, ρ_DS={best.get('rho_DS',0):.3f}")
log.info("\nDone! Download results/ for the benchmark report.")